[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C30_Agent_Harness_Course/02_tool_system/02_tool_system.ipynb)

# 02 · 工具系统（用纯标准库从零搭，MockLLM 端到端）

目标：亲手搭一个完整的**工具系统**——`@tool` 装饰器注册、**JSON Schema** 声明与校验、**分发**（查找/校验/执行/包装）、**错误隔离**（幻觉工具/坏参数/执行异常都变可反馈错误而非崩溃）、**并行工具调用**，全部用 `assert` 验证。

路线：工具与注册表 → JSON Schema 校验 → 分发(含错误隔离) → 幻觉工具/坏参数 → 并行调用(按 id 对位) → 接进 agent 循环 → ✏️ 练习 → 📖 答案 → 🧪 真实 Claude 工具 schema 胶囊。

> 心智模型：**注册表 = 名字→(schema, 函数) 的字典；分发器永远返回结果、从不抛异常**。纯标准库，无需 API key。

## 1 · 工具与注册表：用 `@tool` 装饰器一次性声明

一个工具 = name + description + input_schema + 函数体。我们用装饰器把它们一次声明：函数名→name、docstring→description、显式给 schema。
注册表是一个字典 `name → (schema, fn)`。`export()` 只发 schema（函数体永不出网）。

In [ ]:
class ToolRegistry:
    '''工具注册表：name -> (schema, fn)。export() 给模型看的清单只含 schema。'''
    def __init__(self):
        self._tools = {}
    def tool(self, schema):
        '''装饰器：把一个函数注册成工具。schema 含 name/description/input_schema。'''
        def deco(fn):
            name = schema['name']
            # description 缺省用函数 docstring
            schema.setdefault('description', (fn.__doc__ or '').strip())
            self._tools[name] = (schema, fn)
            return fn
        return deco
    def export(self):
        '''返回给模型的工具清单：只含 schema，绝不含函数体。'''
        return [s for (s, _) in self._tools.values()]
    def __contains__(self, name): return name in self._tools
    def __getitem__(self, name): return self._tools[name]
    def names(self): return list(self._tools)

reg = ToolRegistry()

@reg.tool({'name':'get_weather',
           'input_schema':{'type':'object',
                           'properties':{'city':{'type':'string','description':'城市名'}},
                           'required':['city']}})
def get_weather(city):
    '''查询某城市的当前天气。当用户问天气/气温时使用。'''
    return f'{city}: 晴, 22°C'

@reg.tool({'name':'calculator',
           'input_schema':{'type':'object',
                           'properties':{'expr':{'type':'string','description':'算术表达式'}},
                           'required':['expr']}})
def calculator(expr):
    '''计算一个算术表达式，返回数值结果。'''
    return eval(expr, {'__builtins__':{}}, {})

print('已注册工具:', reg.names())
exported = reg.export()
print('export() 给模型的清单(只含 schema):')
for s in exported: print('  ', s['name'], '->', s['description'])
assert reg.names() == ['get_weather', 'calculator']
assert all('input_schema' in s for s in exported)
# 不变式：export 的工具名 == 能分发的工具名
assert {s['name'] for s in exported} == set(reg.names())
# 安全边界：export 出去的东西里没有任何可调用对象(函数体没出网)
assert all(not callable(v) for s in exported for v in s.values()), 'schema 里不应含函数体'
print('✅ 注册表就绪：装饰器注册、export 只发 schema、函数体不出网')

## 2 · JSON Schema 校验：执行前拦截非法参数

schema 身兼两职：给模型当『使用说明』、给 harness 当『校验规则』。写一个最小校验器，检查：必填项齐全、类型正确、enum 不越界。
校验失败返回 `(False, 原因)` 而非抛异常——这是分发器把坏参数变成可反馈错误的基础。

In [ ]:
def validate(args, schema):
    '''按 JSON Schema 校验参数。返回 (ok, reason)。只实现常用子集。'''
    props = schema.get('properties', {})
    required = schema.get('required', [])
    # 1) 必填项
    for r in required:
        if r not in args:
            return False, f'缺少必填参数: {r}'
    # 2) 类型 + enum
    TYPES = {'string':str, 'integer':int, 'number':(int,float), 'boolean':bool,
             'object':dict, 'array':list}
    for k, v in args.items():
        if k not in props:
            return False, f'未知参数: {k}'
        spec = props[k]
        t = spec.get('type')
        if t in TYPES and not isinstance(v, TYPES[t]):
            # 注意 bool 是 int 的子类，整数校验要排除 bool
            if not (t in ('integer','number') and isinstance(v, bool)):
                if not isinstance(v, TYPES[t]):
                    return False, f'参数 {k} 类型应为 {t}, 实为 {type(v).__name__}'
        if 'enum' in spec and v not in spec['enum']:
            return False, f'参数 {k}={v} 不在允许取值 {spec["enum"]}'
    return True, ''

wschema = reg['get_weather'][0]['input_schema']
print(validate({'city':'北京'}, wschema))         # 合法
print(validate({}, wschema))                      # 缺 city
print(validate({'city':123}, wschema))            # 类型错
print(validate({'city':'北京','foo':1}, wschema)) # 未知参数
assert validate({'city':'北京'}, wschema)[0] is True
assert validate({}, wschema)[0] is False
assert validate({'city':123}, wschema)[0] is False
# enum 校验
es = {'type':'object','properties':{'u':{'type':'string','enum':['c','f']}},'required':['u']}
assert validate({'u':'c'}, es)[0] is True
assert validate({'u':'x'}, es)[0] is False
print('✅ 校验器正确：必填/类型/enum/未知参数都能拦下，且返回原因而非抛异常')

## 3 · 分发：查找→校验→执行→包装（错误全隔离）

分发器是执行中枢，一条流水线：①查找(无则幻觉工具错误) ②校验(不合法不执行) ③执行(隔离异常) ④包装。
**核心契约：dispatch 永远返回一个合法 tool_result，从不抛异常。**

In [ ]:
def _err(call, msg):
    return {'tool_use_id': call['id'], 'content': msg, 'is_error': True}

def dispatch(registry, call):
    '''把一个工具调用落实成执行。永远返回 tool_result，从不抛异常。'''
    name, args = call['name'], call.get('input', {})
    # ① 查找
    if name not in registry:
        return _err(call, f'无此工具 {name}，可用工具: {registry.names()}')
    schema, fn = registry[name]
    # ② 校验
    ok, why = validate(args, schema['input_schema'])
    if not ok:
        return _err(call, f'参数非法: {why}')
    # ③ 执行(隔离异常)
    try:
        result = fn(**args)
    except Exception as e:
        return _err(call, f'工具执行出错: {type(e).__name__}: {e}')
    # ④ 包装
    return {'tool_use_id': call['id'], 'content': str(result), 'is_error': False}

ok = dispatch(reg, {'id':'1','name':'get_weather','input':{'city':'北京'}})
print('正常:', ok)
assert ok['is_error'] is False and '北京' in ok['content']
# 算术工具
r = dispatch(reg, {'id':'2','name':'calculator','input':{'expr':'6*7'}})
assert r['content'] == '42'
print('✅ 分发正常路径跑通：查到函数、校验通过、执行、包装结果')

## 4 · 错误隔离：三类坏调用都不许崩

agent 必须能扛三类坏调用：①模型编了**不存在的工具**(幻觉) ②参数**非法** ③工具**执行时崩溃**。
全部应变成带 `is_error=True` 的结果回灌给模型，让它改正——而非抛异常掀翻整个 agent。

In [ ]:
# ① 幻觉工具：模型编了个不存在的名字
h = dispatch(reg, {'id':'1','name':'send_rocket','input':{}})
print('幻觉工具:', h['content'])
assert h['is_error'] and 'send_rocket' in h['content'] and 'get_weather' in h['content']

# ② 坏参数：缺必填
b = dispatch(reg, {'id':'2','name':'get_weather','input':{}})
print('坏参数:', b['content'])
assert b['is_error'] and '缺少必填' in b['content']

# ③ 执行崩溃：注册一个会抛异常的工具
@reg.tool({'name':'divide','input_schema':{'type':'object',
           'properties':{'a':{'type':'number'},'b':{'type':'number'}},'required':['a','b']}})
def divide(a, b):
    '''a 除以 b。'''
    return a / b

z = dispatch(reg, {'id':'3','name':'divide','input':{'a':1,'b':0}})  # 除零!
print('执行崩溃:', z['content'])
assert z['is_error'] and 'ZeroDivisionError' in z['content']
print('✅ 三类坏调用(幻觉/坏参数/执行崩溃)全部被隔离成可反馈错误，agent 不崩')

## 5 · 并行工具调用：按 tool_use_id 对位

模型一轮可返回**多个**独立 tool_use(如同时查两城天气)。规则：每个调用都要有结果、全部放进**一条** user 消息回传、靠 `tool_use_id` 对位(**绝不按执行顺序硬对**)。我们模拟乱序返回，验证对位正确。

In [ ]:
import concurrent.futures as cf

def dispatch_all(registry, calls, parallel=True):
    '''执行一轮里的多个 tool_use，返回 tool_result 列表(顺序与 calls 一致，靠 id 对回)。'''
    if parallel:
        with cf.ThreadPoolExecutor(max_workers=8) as ex:
            # 提交后按 future 收集，但用 id 重新对位(模拟并发乱序)
            futs = {ex.submit(dispatch, registry, c): c['id'] for c in calls}
            by_id = {}
            for fut in cf.as_completed(futs):     # 完成顺序可能乱
                res = fut.result()
                by_id[res['tool_use_id']] = res
        return [by_id[c['id']] for c in calls]    # 按原 calls 顺序、靠 id 对回
    return [dispatch(registry, c) for c in calls]

@reg.tool({'name':'weather2','input_schema':{'type':'object',
           'properties':{'city':{'type':'string'}},'required':['city']}})
def weather2(city):
    '''带城市差异返回值的天气工具(便于验证对位)。'''
    return {'北京':'北京晴','上海':'上海雨'}.get(city, '未知')

calls = [{'id':'A','name':'weather2','input':{'city':'北京'}},
         {'id':'B','name':'weather2','input':{'city':'上海'}}]
results = dispatch_all(reg, calls, parallel=True)
for r in results: print(r['tool_use_id'], '->', r['content'])
# 关键断言：A 一定对应北京、B 一定对应上海(即便并发乱序也不会错位)
by_id = {r['tool_use_id']: r['content'] for r in results}
assert by_id['A'] == '北京晴' and by_id['B'] == '上海雨', '必须靠 id 对位，不能错位'
# 串行结果应与并行一致
assert dispatch_all(reg, calls, parallel=False) == [by_id_r for by_id_r in results] or True
ser = dispatch_all(reg, calls, parallel=False)
assert {r['tool_use_id']:r['content'] for r in ser} == by_id
print('✅ 并行调用正确：每个调用都有结果、靠 id 对位、串行/并行结果一致')

## 6 · 把工具系统接进 agent 循环

现在把模块 01 的循环 + 本模块的工具系统拼起来，跑一个真用工具的多步任务，并验证**坏调用能被模型『改正』**(下一轮换对工具)。

In [ ]:
class MockLLM:
    def __init__(self, script):
        self.script = list(script); self.calls = 0
    def complete(self, history, tools=None):
        r = self.script[self.calls]; self.calls += 1; return r

def run_agent(llm, registry, task, max_steps=10):
    history = [{'role':'user','content':task}]; trace = []
    for step in range(max_steps):
        resp = llm.complete(history, registry.export())
        history.append({'role':'assistant','content':resp.get('text','')})
        if resp['stop_reason'] == 'end_turn':
            return {'status':'done','answer':resp['text'],'trace':trace}
        results = dispatch_all(registry, resp['tool_calls'], parallel=False)
        for r in results:
            trace.append(('ok' if not r['is_error'] else 'err', r['content']))
        history.append({'role':'user','content':results})
    return {'status':'max_steps','trace':trace}

# 脚本：第1步先调错工具名(幻觉)，看到错误后第2步改对，第3步作答
brain = MockLLM([
    {'stop_reason':'tool_use','text':'','tool_calls':[{'id':'1','name':'wether','input':{'city':'北京'}}]},  # 拼错!
    {'stop_reason':'tool_use','text':'','tool_calls':[{'id':'2','name':'get_weather','input':{'city':'北京'}}]}, # 改对
    {'stop_reason':'end_turn','text':'北京今天晴 22°C'},
])
out = run_agent(brain, reg, '北京天气?')
for t in out['trace']: print('  ', t)
assert out['status'] == 'done'
assert out['trace'][0][0] == 'err', '第1步幻觉工具应是 err'
assert out['trace'][1][0] == 'ok', '第2步改对后应成功'
assert '北京' in out['answer']
print('✅ 工具系统接进循环：幻觉工具被隔离成错误 -> 模型据错改正 -> 完成任务')

---
## ✏️ 练习 1：给注册表加 register 方法

除了装饰器，有时要动态注册工具。给 `ToolRegistry` 实现 `register(name, fn, input_schema, description='')`：
把 `(schema, fn)` 存进去（schema 含 name/description/input_schema）。返回注册后的工具总数。

In [ ]:
def register(self, name, fn, input_schema, description=''):
    # TODO: 组装 schema = {'name':name, 'description':description or fn.__doc__, 'input_schema':input_schema}
    #       存进 self._tools[name] = (schema, fn)；返回 len(self._tools)
    raise NotImplementedError
ToolRegistry.register = register

In [ ]:
# —— 练习 1 自测 ——
r2 = ToolRegistry()
n = r2.register('echo', lambda text: text,
                {'type':'object','properties':{'text':{'type':'string'}},'required':['text']},
                description='原样返回输入')
assert n == 1
assert 'echo' in r2
assert r2.export()[0]['description'] == '原样返回输入'
out = dispatch(r2, {'id':'1','name':'echo','input':{'text':'hi'}})
assert out['is_error'] is False and out['content'] == 'hi'
print('✅ 练习 1 通过：动态 register 正确，注册的工具能被分发')

## ✏️ 练习 2：schema 校验加 number 范围

扩展 `validate2(args, schema)`，在第 2 节基础上支持 `minimum`/`maximum`（数值范围）：
参数有 `minimum` 且 `v < minimum`，或有 `maximum` 且 `v > maximum`，返回 `(False, 原因)`。其余沿用原校验。

In [ ]:
def validate2(args, schema):
    # TODO: 先调用第 2 节的 validate(args, schema)，不通过直接返回
    #       再对每个 args[k] 检查 props[k] 的 minimum/maximum
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
sc = {'type':'object',
      'properties':{'n':{'type':'integer','minimum':1,'maximum':10}},
      'required':['n']}
assert validate2({'n':5}, sc)[0] is True
assert validate2({'n':0}, sc)[0] is False    # < minimum
assert validate2({'n':11}, sc)[0] is False   # > maximum
assert validate2({}, sc)[0] is False         # 缺必填(沿用原校验)
assert validate2({'n':'x'}, sc)[0] is False  # 类型错(沿用原校验)
print('✅ 练习 2 通过：范围校验 + 复用原校验都正确')

## ✏️ 练习 3：把异常工具的报错变得对模型友好

实现 `safe_dispatch(registry, call)`：在第 3 节 dispatch 基础上，当工具**执行抛异常**时，除了标 `is_error=True`，还在 content 里**附上该工具的 schema 提示**(`提示: 期望参数 {required}`)，帮模型下一轮填对。
(幻觉工具/坏参数沿用原 dispatch 行为即可。)

In [ ]:
def safe_dispatch(registry, call):
    # TODO: name 不在 registry -> 同 dispatch 的幻觉工具错误
    #       校验失败 -> 同 dispatch 的坏参数错误
    #       执行抛异常 -> _err(..., f'工具执行出错: {e}. 提示: 期望参数 {required}')
    #       正常 -> 返回结果
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
out = safe_dispatch(reg, {'id':'1','name':'divide','input':{'a':1,'b':0}})
assert out['is_error'] is True
assert 'ZeroDivisionError' in out['content']
assert '提示' in out['content'] and 'a' in out['content'] and 'b' in out['content']
# 正常调用仍正常
good = safe_dispatch(reg, {'id':'2','name':'divide','input':{'a':6,'b':2}})
assert good['is_error'] is False and good['content'] == '3.0'
# 幻觉工具仍被隔离
hal = safe_dispatch(reg, {'id':'3','name':'nope','input':{}})
assert hal['is_error'] is True
print('✅ 练习 3 通过：执行异常时附上 schema 提示，让模型更容易改正')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def register(self, name, fn, input_schema, description=''):
    schema = {'name':name,
              'description':description or (fn.__doc__ or '').strip(),
              'input_schema':input_schema}
    self._tools[name] = (schema, fn)
    return len(self._tools)
ToolRegistry.register = register

In [ ]:
# 练习 2 参考答案
def validate2(args, schema):
    ok, why = validate(args, schema)
    if not ok:
        return ok, why
    props = schema.get('properties', {})
    for k, v in args.items():
        spec = props.get(k, {})
        if 'minimum' in spec and v < spec['minimum']:
            return False, f'参数 {k}={v} 小于最小值 {spec["minimum"]}'
        if 'maximum' in spec and v > spec['maximum']:
            return False, f'参数 {k}={v} 大于最大值 {spec["maximum"]}'
    return True, ''

In [ ]:
# 练习 3 参考答案
def safe_dispatch(registry, call):
    name, args = call['name'], call.get('input', {})
    if name not in registry:
        return _err(call, f'无此工具 {name}，可用工具: {registry.names()}')
    schema, fn = registry[name]
    ok, why = validate(args, schema['input_schema'])
    if not ok:
        return _err(call, f'参数非法: {why}')
    try:
        return {'tool_use_id':call['id'], 'content':str(fn(**args)), 'is_error':False}
    except Exception as e:
        req = schema['input_schema'].get('required', [])
        return _err(call, f'工具执行出错: {type(e).__name__}: {e}. 提示: 期望参数 {req}')

---
## 🧪 真实数据胶囊：本课工具 schema 直接喂给真实 Claude

本课工具 schema 的格式**就是** Claude Messages API 的 `tools` 格式——`{name, description, input_schema}`。
下面把注册表 `export()` 的清单原样传给真实 Claude(有 key 时)，验证它能被 Claude 接受并触发工具调用；无 key 则用 MockLLM 跑通同一套分发逻辑。

> **无 key 也能跑**(回退 MockLLM)；有 `ANTHROPIC_API_KEY` + `anthropic` 时真正调用 `claude-opus-4-8`。

In [ ]:
import os

def claude_tools_from_registry(registry):
    '''本课 export() 的清单几乎就是 Claude 的 tools 格式，直接可用。'''
    return registry.export()   # [{'name','description','input_schema'}, ...]

tools_payload = claude_tools_from_registry(reg)
print('要传给 Claude 的 tools 清单(节选):')
print(' ', tools_payload[0]['name'], '| 必填:', tools_payload[0]['input_schema'].get('required'))
# 校验这份 payload 符合 Claude tools 的基本结构
for t in tools_payload:
    assert set(['name','description','input_schema']) <= set(t), '每个工具需 name/description/input_schema'
    assert t['input_schema']['type'] == 'object', 'input_schema 顶层应为 object'
print('✅ 工具清单符合 Claude Messages API 的 tools 格式，可直接传入')

**🧪 胶囊练习**：实现 `run_once_with_claude(registry, user_msg)`：有 key 时用真实 Claude 跑一轮(`client.messages.create(model='claude-opus-4-8', tools=registry.export(), ...)`，遇到 tool_use 就用本课 `dispatch` 执行)；**无 key 时回退**：用一个会调用注册表里第一个工具的 MockLLM 跑通。返回最终是否成功完成一轮。**绝不因缺 key 报错。**

In [ ]:
def run_once_with_claude(registry, user_msg):
    # TODO: 有 ANTHROPIC_API_KEY 且能 import anthropic ->
    #         client.messages.create(model='claude-opus-4-8', max_tokens=512,
    #             messages=[{'role':'user','content':user_msg}], tools=registry.export())
    #         若 resp.stop_reason=='tool_use': 遍历 tool_use 块用 dispatch 执行(此处可只执行不续轮)
    #         返回 True
    #       无 key -> 用 MockLLM 调注册表第一个工具 + dispatch 跑通, 返回 True。绝不抛异常。
    raise NotImplementedError

In [ ]:
# 自测：本机无论有无 key 都应成功跑通一轮、不报错
ok = run_once_with_claude(reg, '北京天气?')
assert ok is True
print('✅ 胶囊练习通过：工具系统能对接真实 Claude，无 key 自动回退 MockLLM')

In [ ]:
# 📖 胶囊参考答案
def run_once_with_claude(registry, user_msg):
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            resp = client.messages.create(
                model='claude-opus-4-8', max_tokens=512,
                messages=[{'role':'user','content':user_msg}],
                tools=registry.export())
            for blk in resp.content:
                if blk.type == 'tool_use':
                    dispatch(registry, {'id':blk.id,'name':blk.name,'input':blk.input})
            return True
        except Exception:
            pass   # 任何真实调用问题都回退，绝不阻断
    # 回退：MockLLM 调第一个工具
    first = registry.names()[0]
    schema = registry[first][0]['input_schema']
    sample = {k: ('1' if v.get('type')=='string' else 1)
              for k, v in schema.get('properties', {}).items()
              if k in schema.get('required', [])}
    out = dispatch(registry, {'id':'x','name':first,'input':sample})
    return out['is_error'] is False

### 小结
- 工具 = **name + description + input_schema + 函数体**；前三给模型当契约，函数体由 harness 执行、**永不出网**。
- **注册表** = `name → (schema, fn)` 字典；`export()` 只发 schema；不变式：export 的工具名 == 能分发的工具名。
- **JSON Schema** 身兼二职：给模型当使用说明、给 harness 当校验规则；description 写好常比模型更影响调用质量。
- **分发** = 查找→校验→执行→包装；契约是**永远返回 tool_result、从不抛异常**——幻觉工具/坏参数/执行崩溃都变可反馈错误。
- **并行**：每个调用都要有结果、放进**一条** user 消息、靠 `tool_use_id` 对位(绝不按顺序硬对)。
- 本课工具 schema **就是** Claude 的 tools 格式，可直接喂真实 Claude。

下一站：**模块 03 · LLM 适配器** —— 把『大脑』统一成一个接口，MockLLM 与真实 Claude 一行互换。